# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data from `data/raw/`.

In [40]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import geocoder

In [3]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [4]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

Rows and columns: (1694, 53)


,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup
41,3922938,a4498c22-00a2-48a2-b87f-90e6976aa9ed,26,PHX1,2475 W Townley Ave,Phoenix,AZ,85021.0,Maricopa,33.564560,-112.11200,Operating,High,Google Maps and other sources,NaN,NaN,NaN,4,4.0,4.0,Small (0-10 MW),2,NaN,NaN,NaN,NaN,NaN,NaN,46000.0,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://www.expedient.com/data-centers/phoenix/,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.756685e+12,1.771459e+12,NaN
913,3923810,6b38ddd5-0264-4964-9d1a-fa069a8aedec,486,Terra Nexus Ventures Data Center,771 State Rd S-28-331,Camden,SC,29020.0,Kershaw,34.298390,-80.53400,Proposed,High,Google Maps and other sources,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,162.0,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://www.datacenterdynamics.com/en/news/pla...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.758154e+12,1.758154e+12,NaN
1237,3924134,ca92ebde-1d62-44fc-a345-7459c32b8cf0,1790126106451-7608,Sabey Data Centers - Building B,21745 Red Rum Dr,Ashburn,VA,20147.0,Loudoun,39.014496,-77.47998,Proposed,High,Google Maps and other sources,NaN,Sabey,NaN,23,23.0,23.0,Medium (11-50 MW),3,NaN,NaN,NaN,NaN,NaN,NaN,262642.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Crowdsourced,https://opensupplyhub.org/production-locations...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.790035e+12,1.790035e+12,NaN


In [5]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OBJECTID,1694.0,NaN,NaN,NaN,3923743.5,489.159994,3922897.0,3923320.25,3923743.5,3924166.75,3924590.0
GlobalID,1694,1694,e87e9802-1795-43f0-af23-17e816334902,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_id,1694,1694,1779305646902-5353,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_name,1694,1561,H5 Data Center,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,1650,1638,Farm-to-Market 157,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1690,831,Sterling,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,1694,49,VA,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip,1682.0,NaN,NaN,NaN,43739.790131,26461.220655,1085.0,20166.0,30333.5,72192.0,99734.0
county,1691,477,Loudoun,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1694.0,NaN,NaN,NaN,37.604049,4.319988,0.0,33.813575,38.750417,40.06715,70.18478


### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`location_confidence`

In [6]:
# Small fix for location confidence values
data_centers['location_confidence'] = (
    data_centers['location_confidence']
    .apply(lambda x: 'High' if x == 'high' else x)
)

`mw`

In [7]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

,mw,mw_low,mw_high
0,3000,3000.0,3000.0
1,120,120.0,120.0
2,1200,1200.0,1200.0
3,300,300.0,300.0
6,500,500.0,500.0


`power_source`

In [8]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

,power_source,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind
1069,"Natural gas, solar",0,0,0,0,0,0,0,0,1,0,0,0,1,0
650,Solar,0,0,0,0,0,0,0,0,0,0,0,0,1,0
471,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
455,"Natural gas, Batteries",1,0,0,0,0,0,0,0,1,0,0,0,0,0
573,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0


`number_of_buildings`

In [9]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

,number_of_buildings,buildings_low,buildings_high
2,18,18.0,18.0
9,4,4.0,4.0
10,2,2.0,2.0
13,14,14.0,14.0
14,6,6.0,6.0


`cooling_source` + `cooling_type`

In [10]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

,cooling_source,air,water,hybrid_air_water
1132,Hybrid air/water,0,0,1
337,Air,1,0,0
374,Water,0,1,0
559,Water,0,1,0
1091,Water,0,1,0


In [11]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

,cooling_type,closed_loop,fans,open_loop
98,Closed loop,1,0,0
778,Closed loop,1,0,0
756,Closed Loop,1,0,0
559,Closed loop,1,0,0
899,Closed loop,1,0,0


`project_cost`

In [12]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

,project_cost,project_cost_low,project_cost_high
122,$2.6 Billion,2.600000e+09,2.600000e+09
443,$1.5 billion,1.500000e+09,1.500000e+09
897,$1.7 billion,1.700000e+09,1.700000e+09
129,$4.5 billion,4.500000e+09,4.500000e+09
70,$78 million,7.800000e+07,7.800000e+07


### FracTracker Crypto Mining Operations

In [13]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

Rows and columns: (302, 18)


,Name,City,State,Address,Company,Probable power source,Links,Status,lat,long,Precise Location?,MapCat,GEM_links,Other updates,Date,MW (NYTimes),Unnamed: 16,Unnamed: 17
103,Riot Platforms greenfield site,Paducah,KY,"Coleman Rd, Paducah, KY 42001",Riot,NaN,https://markets.businessinsider.com/news/stock...,Proposed,37.088592,-88.676158,No,Proposed,NaN,NaN,7/25/2024,60 MW-->150 MW,NaN,NaN
193,Longhorn Pad C well pad,Weedville,PA,"Swede Farm Rd, Weedville, PA 15868",Diversified Production LLC,Wellhead/Fracked gas,https://www.pacodeandbulletin.gov/secure/pabul...,Operating,41.295556,-78.453222,Yes,Operating / under construction,NaN,NaN,6/15/2024,NaN,NaN,NaN
4,Interstate Holdings Blockchain,Cabot,AR,"Grayhawk Rd, Cabot, AR 72023",Interstate Holdings Blockchain,NaN,https://arktimes.com/arkansas-blog/2024/12/06/...,Proposed,34.967105,-92.052562,Yes,Proposed,NaN,NaN,2/3/2025,NaN,NaN,NaN


In [14]:
crypto.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,302,250,CleanSpark,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,283,235,Dalton,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,302,41,GA,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Address,194,189,"430 Davis Rd, Vidalia, GA 30474",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Company,283,174,CleanSpark,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Probable power source,163,36,Unknown,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Links,299,263,https://www.wbir.com/article/tech/east-tenness...,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,301,11,Operating,138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,302.0,NaN,NaN,NaN,37.830255,5.191345,25.454595,34.456008,36.356141,41.261212,69.71761
long,302.0,NaN,NaN,NaN,-128.525789,651.005258,-11402.57827,-98.202512,-88.774187,-82.477328,-69.170992


### FracTracker Crypto Mining Operations - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `Probable power source` | Power source values use inconsistent naming conventions. | Normalize them to the FracTracker data center categories using keyword mapping. |
| `Precise Location?` | Values represented as 'Yes' / 'No' strings. | Update to a boolean representation. |
| `MW (NYTimes)` | A range of values including '100 MW', '100 MW+', '100 MW --> 200 MW'. | Standardize to `mw_low` and `mw_high` to keep consistent representation in both datasets. |

`Probable power source`

In [15]:
# Reuse power sources from FracTracker data centers
power_sources = [
    "batteries", "biomass", "coal", "fuel_cells",
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid",
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel",
    "renewable", "solar", "wind"
]

# Mapping given sources to predefined power sources
def normalize_sources(source):
    if pd.isna(source):
        return []

    source = source.lower()
    found_sources = set()

    # Keyword mappings
    keyword_map = {
        "battery": "batteries",
        "biomass": "biomass",
        "coal": "coal",
        "fuel cell": "fuel_cells",
        "geothermal": "geothermal",
        "hydro": "hydroelectric",
        "gas": "natural_gas",
        "nuclear": "nuclear",
        "diesel": "oil_diesel",
        "oil": "oil_diesel",
        "solar": "solar",
        "wind": "wind",
        "renewable": "renewable",
        "wte": "renewable",
    }

    for keyword, category in keyword_map.items():
        if keyword in source:
            found_sources.add(category)

    # Grid
    if "grid" in source:
        found_sources.add("grid_unspecified_mix")

    # Grid + another identified source = hybrid
    non_grid_sources = found_sources - {"grid_unspecified_mix"}

    if "grid_unspecified_mix" in found_sources and non_grid_sources:
        found_sources.add("hybrid_onsite_and_grid")

    return sorted(found_sources)


crypto["norm_power_source"] = crypto["Probable power source"].apply(normalize_sources)
crypto.loc[crypto["norm_power_source"].str.len() > 0][["norm_power_source"]].sample(5)

,norm_power_source
233,[solar]
132,[natural_gas]
105,[biomass]
199,[natural_gas]
104,[grid_unspecified_mix]


`Precise Location?`

In [16]:
# Change precise location to a boolean
crypto["Precise Location?"] = crypto["Precise Location?"].apply(lambda x: x == 'Yes')
crypto[['Precise Location?']].sample(5)

,Precise Location?
14,True
115,True
56,False
282,True
170,True


`MW (NYTimes)`

In [17]:
# Function for standardizing mw values
def standardize_mw(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).lower().strip()

    # Fix typo
    text = text.replace("nw", "mw")

    # Extract numbers followed by MW or GW
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(mw|gw)', text)

    if not matches:
        return np.nan, np.nan

    values_mw = []

    for number, unit in matches:
        number = float(number)

        # Convert to MW
        if unit == "gw":
            number *= 1000

        values_mw.append(number)

    return min(values_mw), max(values_mw)


# Add new features to handle ranges
crypto[["mw_low", "mw_high"]] = pd.DataFrame(
    crypto["MW (NYTimes)"].apply(standardize_mw).tolist(),
    index=crypto.index
)

crypto.loc[crypto['MW (NYTimes)'].notna()][["MW (NYTimes)", "mw_low", "mw_high"]].sample(5)

,MW (NYTimes),mw_low,mw_high
54,15 MW,15.0,15.0
281,100 MW,100.0,100.0
219,60 MW,60.0,60.0
94,250 MW,250.0,250.0
101,25 MW--> 55 MW,25.0,55.0


### Address Standardization & Comparison 

Addresses are standardized across both datasets to make facility matching more reliable. Common street names, directions, and unit types are converted to consistent abbreviations, while special characters and extra whitespace are removed. The crypto dataset sometimes stores the city, state, and ZIP code inside the Address field. These values are separated so the final location fields are: 

- **Crypto**: `Address`, `City`, `State`, `Zip` 
- **Data Centers**: `address`, `city`, `state`, `zip`

#### Address Abbreviations

In [18]:
ADDRESS_ABBREVIATIONS = {
    # Directions
    'north': 'n',
    'south': 's',
    'east': 'e',
    'west': 'w',
    'northeast': 'ne',
    'northwest': 'nw',
    'southeast': 'se',
    'southwest': 'sw',

    # Street suffixes
    'street': 'st',
    'avenue': 'ave',
    'road': 'rd',
    'drive': 'dr',
    'lane': 'ln',
    'boulevard': 'blvd',
    'circle': 'cir',
    'court': 'ct',
    'place': 'pl',
    'parkway': 'pkwy',
    'highway': 'hwy',
    'trail': 'trl',
    'terrace': 'ter',
    'square': 'sq',
    'crossing': 'xing',
    'center': 'ctr',
    'centre': 'ctr',
    'point': 'pt',
    'ridge': 'rdg',
    'run': 'run',
    'view': 'vw',
    'village': 'vlg',
    'station': 'sta',
    'landing': 'lndg',
    'junction': 'jct',
    'extension': 'ext',
    'expressway': 'expy',
    'freeway': 'fwy',
    'turnpike': 'tpke',
    'causeway': 'cswy',
    'crescent': 'cres',
    'cove': 'cv',
    'fork': 'frk',
    'garden': 'gdn',
    'gardens': 'gdns',
    'grove': 'grv',
    'harbor': 'hbr',
    'harbour': 'hbr',
    'heights': 'hts',
    'hill': 'hl',
    'hills': 'hls',
    'island': 'is',
    'islands': 'iss',
    'knoll': 'knl',
    'knolls': 'knls',
    'lake': 'lk',
    'lakes': 'lks',
    'manor': 'mnr',
    'manors': 'mnrs',
    'meadow': 'mdw',
    'meadows': 'mdws',
    'mount': 'mt',
    'mountain': 'mtn',
    'orchard': 'orch',
    'pine': 'pne',
    'pines': 'pnes',
    'plaza': 'plz',
    'port': 'prt',
    'prairie': 'pr',
    'radial': 'radl',
    'river': 'riv',
    'shore': 'shr',
    'shores': 'shrs',
    'spring': 'spg',
    'springs': 'spgs',
    'summit': 'smt',
    'union': 'un',
    'valley': 'vly',
    'valleys': 'vlys',
    'well': 'wl',
    'wells': 'wls',
    
    # Unit Overrides
    'apartment': 'apt',
    'apt.': 'apt',
    'suite': 'ste',
    'ste.': 'ste',
    'unit': 'unit',
    '#': 'unit',
    'building': 'bldg',
    'floor': 'fl',
    'room': 'rm',
    'department': 'dept'
}

#### Helper Functions

These functions are used to standardize addresses by removing special characters and extra whitespace and replacing common address terms with predefined abbreviations. In the crypto dataset, the `Address` field sometimes also contains the city, state, and ZIP code. A second function, `remove_city_state`, removes this duplicated location information, while ZIP codes are extracted into a separate column.

In [19]:
def standardize_address(address):
    if pd.isna(address):
        return np.nan
    
    address = address.lower().strip()
    
    # Remove special characters
    address = re.sub(r'[^a-z0-9\s]', '', address)
    address_parts = address.split()
    
    # Update addresses using abbreviations
    cleaned_parts = [
        ADDRESS_ABBREVIATIONS.get(part, part)
        for part in address_parts
    ]
    
    return " ".join(cleaned_parts)


def remove_city_state(address, city, state):
    if pd.isna(address):
        return np.nan

    address = str(address)

    # Remove ZIP from address
    address = re.sub(r'\s*,?\s*\d{5}\s*$', '', address)

    # Remove State from address
    if pd.notna(state):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(state))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    # Remove City from address
    if pd.notna(city):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(city))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    return address.strip(' ,')

In [20]:
# Extract 5-digit zip if exists
crypto['Zip'] = crypto['Address'].str.extract(r'(\d{5})\s*$')[0]
crypto['Zip'] = crypto['Zip'].replace('', np.nan).astype('Int64') # missing zip

# Remove city, state, and zip from addresses in crypto
crypto['Address'] = crypto.apply(
    lambda row: remove_city_state(
        row['Address'],
        row['City'],
        row['State']
    ),
    axis=1
)

# Applying the transformations to the addresses in both datasets
crypto['Address'] = crypto['Address'].apply(standardize_address)
data_centers['address'] = data_centers['address'].apply(standardize_address)


# data_centers[['address', 'city', 'state', 'zip']].sample(5)
crypto[['Address', 'City', 'State', 'Zip']].sample(5)

,Address,City,State,Zip
47,airport rd,Baldwin,GA,30511
105,NaN,Hopkinsville,KY,<NA>
175,3807 highland ave,Niagara Falls,NY,14305
263,NaN,Odessa,TX,<NA>
217,NaN,New Tazewell,TN,<NA>


#### Address Comparison

The standardized datasets are merged using exact matches on address, city, and state to identify facilities that appear in both datasets. This allows us to identify data centers associated with known crypto mining locations, providing more information for later analysis. Using exact address matching, 19 matching records were identified between the two datasets.

In [21]:
# Find matching addresses between the datasets
matches = crypto[crypto['Address'].notna()].merge(
    data_centers[data_centers['address'].notna()],
    left_on=['Address', 'City', 'State'],
    right_on=['address', 'city', 'state'],
    how='inner',
    suffixes=('_crypto', '_dc')
)

print(f"{len(matches)} matches found between the crypto and data centers datatsets.")
matches[['facility_name', 'Name', 'lat_crypto', 'long_crypto', 'lat_dc', 'long_dc', 'address', 'city', 'state']]

19 matches found between the crypto and data centers datatsets.


,facility_name,Name,lat_crypto,long_crypto,lat_dc,long_dc,address,city,state
0,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific Whitfield County facility (sel...,34.713160,-84.951430,34.713160,-84.951430,2205 industrial s rd,Dalton,GA
1,Riot 1,Block Mining McCracken County facility,37.088310,-88.703050,37.088310,-88.703050,5600 commerce dr,Paducah,KY
2,Riot 2,Riot Platforms greenfield site,37.088592,-88.676158,37.088593,-88.676155,coleman rd,Paducah,KY
3,Hyperscale Data Michigan Data Center,Michigan Data Center,41.993228,-86.098603,41.992455,-86.098976,415 e pr ronde st,Dowagiac,MI
4,CoreWeave/ Applied Digital,Applied Digital Stutsman County facility,47.027383,-98.679666,47.027080,-98.679600,2671 hwy 20 se,Jamestown,ND
5,Applied Digital: Polaris Forge 1,"Applied Digital, being repurposed from crypto ...",46.010438,-98.570800,46.012590,-98.570700,9685 87th ave se,Ellendale,ND
6,Vulcan Infrastructure and Power Inc (formerly ...,Greenidge Generation Yates County facility,42.676870,-76.944910,42.676870,-76.944910,590 plant rd,Dresden,NY
7,BlockFusion aka North East Data Center LLC,North East Data LLC Niagara facility,43.082864,-79.003495,43.083187,-79.003170,5380 frontier ave,Niagara Falls,NY
8,9Lives Data Center,Nine Lives Digital Group,43.123208,-79.040614,43.123890,-79.041180,3807 highland ave,Niagara Falls,NY
9,North Tonawanda Data Center,Digipower X Fortistar Niagara County facility,43.060860,-78.853760,43.060860,-78.853760,1070 erie ave,North Tonawanda,NY


### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [22]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

Rows and columns: (17349, 42)


,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,Longitude,NERC Region,Balancing Authority Code,Balancing Authority Name,Name of Water Source,Primary Purpose (NAICS Code),Regulatory Status,Sector,Sector Name,FERC Cogeneration Status,FERC Cogeneration Docket Number,FERC Small Power Producer Status,FERC Small Power Producer Docket Number,FERC Exempt Wholesale Generator Status,FERC Exempt Wholesale Generator Docket Number,Ash Impoundment?,Ash Impoundment Lined?,Ash Impoundment Status,Transmission or Distribution System Owner,Transmission or Distribution System Owner ID,Transmission or Distribution System Owner State,Grid Voltage (kV),Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
8339,60438,Paynesville Community Solar,60715,Paynesville Community Solar,18982 293rd Ave,Paynesville,MN,56362,Stearns,45.402543,-94.715472,MRO,MISO,Midcontinent Independent Transmission System O...,NaN,22,NR,2,IPP Non-CHP,N,NaN,Y,"17-88-000, 17-89-000, 17-90-000, 17-91-000, 17...",N,NaN,NaN,NaN,NaN,Northern States Power Co - Minnesota,13781,MN,34.5,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7084,58135,Ecos Energy LLC,59267,Mission Solar LLC,235 Vassar Avenue,Merced,CA,95341,Merced,37.267778,-120.4825,WECC,CISO,California Independent System Operator,NaN,22,NR,2,IPP Non-CHP,N,NaN,Y,QF15-150,N,NaN,N,X,NaN,Pacific Gas & Electric Co,14328,CA,21,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1257,4226,Consolidated Edison Co-NY Inc,2503,59th Street,850 12th Avenue,New York,NY,10019,New York,40.771285,-73.99115,NPCC,NYIS,New York Independent System Operator,Hudson River,22,RE,1,Electric Utility,N,NaN,N,NaN,N,NaN,NaN,X,NaN,Consolidated Edison Co-NY Inc,4226,NY,13.8,,,N,NaN,TRANSCONTINENTAL GAS PIPELINE,NaN,NaN,NaN,N,NaN


In [23]:
plants.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,17349.0,NaN,NaN,NaN,47106.087844,23529.704706,7.0,18642.0,60301.0,64545.0,68045.0
Utility Name,17349,6935,"Altus Power America Management, LLC",247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,17349.0,NaN,NaN,NaN,52099.079601,22868.185984,1.0,55701.0,61103.0,65921.0,70452.0
Plant Name,17349,17345,Viola,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Street Address,17229,15510,TBD,789,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,17314,6015,Lancaster,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,17349,51,CA,2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip,17349.0,8431.0,93501.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,17326,1542,Kern,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,17349,16666,,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [24]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

Rows and columns: (27919, 73)


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,Ownership,Duct Burners,Can Bypass Heat Recovery Steam Generator?,RTO/ISO LMP Node Designation,RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Uprate or Derate Completed During Year,Month Uprate or Derate Completed,Year Uprate or Derate Completed,Status,Synchronized to Transmission Grid,Operating Month,Operating Year,Planned Retirement Month,Planned Retirement Year,Associated with Combined Heat and Power System,Sector Name,Sector,Topping or Bottoming,Energy Source 1,Energy Source 2,Energy Source 3,Energy Source 4,Energy Source 5,Energy Source 6,Startup Source 1,Startup Source 2,Startup Source 3,Startup Source 4,Solid Fuel Gasification System?,Carbon Capture Technology?,Turbines or Hydrokinetic Buoys,Time from Cold Shutdown to Full Load,Fluidized Bed Technology?,Pulverized Coal Technology?,Stoker Technology?,Other Combustion Technology?,Subcritical Technology?,Supercritical Technology?,Ultrasupercritical Technology?,Planned Net Summer Capacity Uprate (MW),Planned Net Winter Capacity Uprate (MW),Planned Uprate Month,Planned Uprate Year,Planned Net Summer Capacity Derate (MW),Planned Net Winter Capacity Derate (MW),Planned Derate Month,Planned Derate Year,Planned New Prime Mover,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
20415,65411,Deriva Energy,62141.0,Rambler,TX,Tom Green,RMBLR,Solar Photovoltaic,PV,NaN,W,X,X,NaN,NaN,200.0,,200,200,,N,,,OP,X,6.0,2020.0,,,N,IPP Non-CHP,2.0,X,SUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,NaN,NaN,NaN
10171,11295,Los Angeles County-San Gabriel,54017.0,San Gabriel Hydro Project,CA,Los Angeles,U2,Conventional Hydroelectric,HY,NaN,S,X,X,NaN,NaN,1.1,0.9,1.1,1,0.1,N,,,OP,X,12.0,1987.0,,,N,IPP Non-CHP,2.0,X,WAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,0,10M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,N,,,N,NaN,NaN
23078,57416,"Acciona Energy USA Global, LLC",64660.0,"AEUG Union Solar, LLC",OH,Union,AUS,Solar Photovoltaic,PV,NaN,S,X,X,NaN,NaN,325.0,0.95,325,325,,N,,,OP,X,11.0,2024.0,,,N,IPP Non-CHP,2.0,X,SUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,NaN


In [25]:
generators.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,27919.0,5853.0,55938.0,329.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Utility Name,27918,5842,Industrial Power Generating Company LLC,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,27918.0,NaN,NaN,NaN,42375.107386,26782.287436,1.0,7146.0,56794.5,62643.75,70438.0
Plant Name,27918,14325,Edison Sault,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,27918,51,CA,3161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,27918,1479,Los Angeles,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Generator ID,27917,10890,1,2923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technology,27918,27,Solar Photovoltaic,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prime Mover,27918,18,PV,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit Code,2180,310,CC1,770,NaN,NaN,NaN,NaN,NaN,NaN,NaN


EIA-860 Defining Energy Source Features

In [37]:
energy_sources = pd.read_csv(
    DATA_DIR / "eia860_2025" / "eia_860_energy_source_codes.csv",
)
energy_sources.rename(columns={'Energy Source Code': 'Energy Source 1'}, inplace=True)
energy_sources.sample(3)

,Energy Source 1,Category,Description
24,AB,Renewables and Biomass,Agricultural By-Products / Straw / Energy Crops
8,OG,Coal and Fossil Fuels,Other Gas
20,GEO,Renewables and Biomass,Geothermal


In [112]:
#generators['Energy Source 1'].unique()
generator_features = generators[['State','County','Energy Source 1','Technology','Time from Cold Shutdown to Full Load','Summer Capacity (MW)','Winter Capacity (MW)']].copy()
# Dropping description to create categorical feature from Energy Source
generator_features_full = pd.merge(generator_features, energy_sources, on='Energy Source 1', how='left').drop('Description', axis=1)
generator_features_full.rename(columns={'Category':'Energy Category','Time from Cold Shutdown to Full Load':'Time to Full Load'}, inplace=True)
# Clean Time to Full Load from string to numerical value
generator_features_full.sample(15)

,State,County,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW),Energy Category
22905,MN,Chisago,SUN,Solar Photovoltaic,NaN,1,1,Renewables and Biomass
14230,OR,Umatilla,WND,Onshore Wind Turbine,NaN,6.6,6.6,Renewables and Biomass
17596,TX,Jim Hogg,WND,Onshore Wind Turbine,NaN,78,78,Renewables and Biomass
15104,MN,Hennepin,WAT,Conventional Hydroelectric,10M,0.6,0.6,Renewables and Biomass
19267,NY,Albany,WAT,Conventional Hydroelectric,10M,1.5,1.5,Renewables and Biomass
10598,NY,Oswego,NG,Natural Gas Fired Combined Cycle,12H,174.54,208.66,Coal and Fossil Fuels
15246,CA,Sacramento,SUN,Solar Photovoltaic,NaN,5,5,Renewables and Biomass
12054,AL,Calhoun,NG,Natural Gas Fired Combustion Turbine,12H,176,195,Coal and Fossil Fuels
25813,WI,Chippewa,SUN,Solar Photovoltaic,NaN,1.5,1.5,Renewables and Biomass
13935,TX,Erath,WND,Onshore Wind Turbine,NaN,52.8,52.8,Renewables and Biomass


Time to Full Load in Minutes Clean Up

In [113]:
def time_in_min(stringval):
    val = str(stringval).lower().strip()
    # Match digits versus text
    regex = re.match(r"(\d+)(\s*[a-z]+)", val)
    if not regex:
        return None

    if regex.group(2) in ['m']:
        return int(regex.group(1))
    elif regex.group(2) in ['h']:
        return int(regex.group(1))*60
    else:
        return None

In [114]:
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].apply(time_in_min)


In [115]:
generator_features_full.describe()
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].fillna(generator_features_full['Time to Full Load'].mean().round(1))

One-Hot Encoding for Energy

In [116]:
one_hot_energy = pd.get_dummies(generator_features_full['Energy Source 1'])
generator_features_full = generator_features_full.join(one_hot_energy)
generator_features_full.sample(5)

,State,County,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW),Energy Category,AB,BFG,BIT,BLQ,DFO,GEO,H2,JF,KER,LFG,LIG,MSW,MWH,NG,NUC,OBG,OBL,OBS,OG,OTH,PC,PG,PUR,RC,RFO,SGC,SUB,SUN,WAT,WC,WDL,WDS,WH,WND,WO
13340,AR,Craighead,NG,Natural Gas Fired Combustion Turbine,60.0,20,23,Coal and Fossil Fuels,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8256,NJ,WARREN,NG,Natural Gas Fired Combined Cycle,60.0,8.8,10.7,Coal and Fossil Fuels,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
6417,AK,Northwest Arctic,DFO,Petroleum Liquids,10.0,0.4,0.4,Petroleum and Liquids,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
25479,MD,Howard,SUN,Solar Photovoltaic,160.9,2,2,Renewables and Biomass,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
10705,NH,Coos,WAT,Conventional Hydroelectric,10.0,0.8,0.8,Renewables and Biomass,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False


### EIA-861M Retail Sales of Electricity


In [117]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

Rows and columns: (10099, 24)


,Year,Month,State,Data Status,Thousand Dollars,Megawatthours,Count,Cents/kWh,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1,Thousand Dollars.2,Megawatthours.2,Count.2,Cents/kWh.2,Thousand Dollars.3,Megawatthours.3,Count.3,Cents/kWh.3,Thousand Dollars.4,Megawatthours.4,Count.4,Cents/kWh.4
377,2025,11.0,MD,Preliminary,415446.86,1946745.3,2470725.0,21.34,318874.91,2063431.7,262970.0,15.45,36019.2,288762.69,9391.0,12.47,5409.11,35950.10,5.0,15.05,775750.07,4334889.8,2743091.0,17.90
4255,2019,7.0,MI,Final,647466.00,3958183.9,4382269.0,16.36,425191.79,3721652.7,542360.0,11.42,183651.8,2560887.90,5806.0,7.17,72.85,647.82,2.0,11.25,1256382.40,10241372.0,4930437.0,12.27
9745,2010,7.0,CA,Final,1217813.00,8105267.0,12990786.0,15.02,1686018.00,11041978.0,1823437.0,15.27,488616.0,4398786.00,75655.0,11.11,6073.00,70279.00,14.0,8.64,3398520.00,23616311.0,14889892.0,14.39


In [118]:
sales.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,10099.0,18.0,2025.0,612.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,10098.0,NaN,NaN,NaN,6.409091,3.451026,1.0,3.0,6.0,9.0,12.0
State,10098,51,AK,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Status,10098,2,Final,9180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Thousand Dollars,10098.0,NaN,NaN,NaN,320957.591635,371619.450947,13064.0,78169.825,217794.035,402375.0,3266301.5
Megawatthours,10098.0,NaN,NaN,NaN,2353244.461886,2549429.123467,104574.0,594574.7925,1704852.8,3130227.025,22481421.0
Count,10098.0,NaN,NaN,NaN,2630209.189344,2730303.6032,226548.0,802266.75,1970445.0,3101617.5,14730091.0
Cents/kWh,10098.0,NaN,NaN,NaN,14.236286,5.232916,6.73,11.07,12.69,15.5875,52.72
Thousand Dollars.1,10098.0,NaN,NaN,NaN,251066.79199,314470.995416,20778.0,66681.4875,165946.025,313538.0625,3327766.4
Megawatthours.1,10098.0,NaN,NaN,NaN,2237676.566305,2411765.932939,126343.74,641958.2175,1621191.95,2820912.175,16965023.0


In [122]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 10099 entries, 0 to 10098
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Year                10099 non-null  object 
 1   Month               10098 non-null  float64
 2   State               10098 non-null  str    
 3   Data Status         10098 non-null  str    
 4   Thousand Dollars    10098 non-null  float64
 5   Megawatthours       10098 non-null  float64
 6   Count               10098 non-null  float64
 7   Cents/kWh           10098 non-null  float64
 8   Thousand Dollars.1  10098 non-null  float64
 9   Megawatthours.1     10098 non-null  float64
 10  Count.1             10098 non-null  float64
 11  Cents/kWh.1         10098 non-null  float64
 12  Thousand Dollars.2  10098 non-null  float64
 13  Megawatthours.2     10098 non-null  float64
 14  Count.2             10098 non-null  float64
 15  Cents/kWh.2         10098 non-null  float64
 16  Thousand Dollar

In [124]:
sales['Year'] = sales['Year'].astype(str)

In [126]:
# Looking at 2025 only and only .1 since that is Commercial
# EIA classifies data centers as commercial customers
sales_2025 = sales[sales['Year']=='2025'].copy()
sales_2025 = sales_2025[['Year','Month','State','Thousand Dollars.1','Megawatthours.1','Count.1','Cents/kWh.1']]
sales_2025.sample(5)

,Year,Month,State,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1
571,2025,7.0,GA,709093.32,5678953.30,627934.0,12.49
618,2025,6.0,CT,231538.23,1086843.50,156334.0,21.30
551,2025,8.0,SD,53404.84,488461.95,80260.0,10.93
543,2025,8.0,NV,146909.84,1441535.90,178650.0,10.19
437,2025,10.0,NE,108311.09,1228294.50,165054.0,8.82


In [137]:
# Group by State, Summer/Winter AVG
# Summer/Winter to match generator dataset capacity values
sales_2025['Season'] = sales_2025['Month'].apply(
    lambda x: 'summer' if 4 <= x <= 9 else 'winter'
)
grouped_sales = sales_2025.groupby(by=['State','Season'], as_index=False).mean(numeric_only=True).sort_values(by='State', ascending=True).drop(columns=['Month'])
grouped_sales.head(10)

,State,Season,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1
0,AK,summer,4.818776e+04,2.120679e+05,5.840083e+04,22.718333
1,AK,winter,5.055212e+04,2.303088e+05,5.806733e+04,21.958333
2,AL,summer,2.980218e+05,2.057032e+06,3.959620e+05,14.486667
3,AL,winter,2.590876e+05,1.779264e+06,3.959318e+05,14.563333
4,AR,summer,1.166311e+05,1.070972e+06,2.092887e+05,10.906667
5,AR,winter,9.688299e+04,9.132678e+05,2.091840e+05,10.611667
6,AZ,summer,4.453811e+05,3.455614e+06,3.505910e+05,12.855000
7,AZ,winter,3.346111e+05,2.801567e+06,3.505335e+05,11.940000
8,CA,summer,2.682709e+06,9.741230e+06,1.776620e+06,27.353333
9,CA,winter,2.304272e+06,9.178307e+06,1.766771e+06,25.031667


### EIA-930 Balance Data

In [91]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

Rows and columns: (267218, 65)


,Balancing Authority,Data Date,Hour Number,Local Time at End of Hour,UTC Time at End of Hour,Demand Forecast (MW),Demand (MW),Net Generation (MW),Total Interchange (MW),Sum(Valid DIBAs) (MW),Demand (MW) (Imputed),Net Generation (MW) (Imputed),Total Interchange (MW) (Imputed),Demand (MW) (Adjusted),Net Generation (MW) (Adjusted),Total Interchange (MW) (Adjusted),Net Generation (MW) from Coal,Net Generation (MW) from Natural Gas,Net Generation (MW) from Nuclear,Net Generation (MW) from All Petroleum Products,Net Generation (MW) from Hydropower Excluding Pumped Storage,Net Generation (MW) from Pumped Storage,Net Generation (MW) from Solar without Integrated Battery Storage,Net Generation (MW) from Solar with Integrated Battery Storage,Net Generation (MW) from Wind without Integrated Battery Storage,Net Generation (MW) from Wind with Integrated Battery Storage,Net Generation (MW) from Battery Storage,Net Generation (MW) from Other Energy Storage,Net Generation (MW) from Unknown Energy Storage,Net Generation (MW) from Geothermal,Net Generation (MW) from Other Fuel Sources,Net Generation (MW) from Unknown Fuel Sources,Net Generation (MW) from Coal (Imputed),Net Generation (MW) from Natural Gas (Imputed),Net Generation (MW) from Nuclear (Imputed),Net Generation (MW) from All Petroleum Products (Imputed),Net Generation (MW) from Hydropower Excluding Pumped Storage (Imputed),Net Generation (MW) from Pumped Storage (Imputed),Net Generation (MW) from Solar without Integrated Battery Storage (Imputed),Net Generation (MW) from Solar with Integrated Battery Storage (Imputed),Net Generation (MW) from Wind without Integrated Battery Storage (Imputed),Net Generation (MW) from Wind with Integrated Battery Storage (Imputed),Net Generation (MW) from Battery Storage (Imputed),Net Generation (MW) from Other Energy Storage (Imputed),Net Generation (MW) from Unknown Energy Storage (Imputed),Net Generation (MW) from Geothermal (Imputed),Net Generation (MW) from Other Fuel Sources (Imputed),Net Generation (MW) from Unknown Fuel Sources (Imputed),Net Generation (MW) from Coal (Adjusted),Net Generation (MW) from Natural Gas (Adjusted),Net Generation (MW) from Nuclear (Adjusted),Net Generation (MW) from All Petroleum Products (Adjusted),Net Generation (MW) from Hydropower Excluding Pumped Storage (Adjusted),Net Generation (MW) from Pumped Storage (Adjusted),Net Generation (MW) from Solar without Integrated Battery Storage (Adjusted),Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind without Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind with Integrated Battery Storage (Adjusted),Net Generation (MW) from Battery Storage (Adjusted),Net Generation (MW) from Other Energy Storage (Adjusted),Net Generation (MW) from Unknown Energy Storage (Adjusted),Net Generation (MW) from Geothermal (Adjusted),Net Generation (MW) from Other Fuel Sources (Adjusted),Net Generation (MW) from Unknown Fuel Sources (Adjusted),Region
201796,SIKE,11/04/2025,13,11/04/2025 1:00:00 PM,11/04/2025 7:00:00 PM,NaN,NaN,21.0,21.0,22.0,NaN,NaN,NaN,NaN,21.0,21.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MIDW
239136,TPWR,07/27/2025,3,07/27/2025 3:00:00 AM,07/27/2025 10:00:00 AM,331.0,321.0,73.0,-248.0,-248.0,NaN,NaN,NaN,321.0,73.0,-248.0,NaN,NaN,NaN,NaN,73.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NW
140793,NWMT,12/09/2025,7,12/09/2025 7:00:00 AM,12/09/2025 2:00:00 PM,1369.0,1362.0,2458.0,1096.0,1096.0,NaN,NaN,NaN,1362.0,2458.0,1096.0,743.0,197.0,NaN,58.0,371.0,NaN,0.0,NaN,1089.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,743.0,197.0,NaN,58.0,371.0,NaN,0.0,NaN,1089.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [92]:
balance.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,267218,61,AECI,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,267218,184,11/02/2025,1490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,267218.0,NaN,NaN,NaN,12.502339,6.923663,1.0,7.0,13.0,19.0,25.0
Local Time at End of Hour,267218,4416,11/02/2025 1:00:00 AM,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,267218,4420,07/01/2025 8:00:00 AM,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand Forecast (MW),233431.0,NaN,NaN,NaN,9465.17759,19303.64298,-227966.0,791.0,2449.0,6493.0,156738.0
Demand (MW),233639.0,NaN,NaN,NaN,9503.464246,19220.239208,-39388.0,815.0,2544.0,6665.0,810948.0
Net Generation (MW),266717.0,NaN,NaN,NaN,8304.861227,18525.972929,-2576.0,547.0,1759.0,5935.0,810705.0
Total Interchange (MW),266739.0,NaN,NaN,NaN,-78.283611,1344.438016,-11031.0,-496.0,-8.0,387.0,40651.0
Sum(Valid DIBAs) (MW),265658.0,NaN,NaN,NaN,-60.813783,1416.714082,-57015.0,-505.0,-13.0,374.0,40651.0


### EIA-930 Subregion Data

In [93]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

Rows and columns: (366596, 7)


,Balancing Authority,Data Date,Hour Number,Sub-Region,Demand (MW),Local Time at End of Hour,UTC Time at End of Hour
348069,SWPP,11/16/2025,15,INDN,87.0,11/16/2025 3:00:00 PM,11/16/2025 9:00:00 PM
133034,NYIS,09/07/2025,23,ZONE,716.0,09/07/2025 11:00:00 PM,09/08/2025 3:00:00 AM
307996,SWPP,08/10/2025,10,WR,3654.0,08/10/2025 10:00:00 AM,08/10/2025 3:00:00 PM


In [94]:
subregion.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,366596,8,PJM,88340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,366596,184,11/02/2025,2060,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,366596.0,NaN,NaN,NaN,12.502319,6.923647,1.0,7.0,13.0,19.0,25.0
Sub-Region,366596,83,PGAE,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand (MW),365441.0,NaN,NaN,NaN,3942.233748,4968.20665,-4.0,814.0,1889.0,4780.0,32436.0
Local Time at End of Hour,366596,4416,11/02/2025 1:00:00 AM,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,366596,4420,07/01/2025 8:00:00 AM,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save cleaned data

Save the cleaned data centers to `data/processed/` so later notebooks can load them without re-running this one.

In [95]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

Saved: fractracker_data_centers_clean.csv with 1701 rows and 77 columns
